In [ ]:
import os
import sys
import numpy as np


if os.path.basename(os.getcwd()).lower() == "notebooks":
    ROOT_DIR = os.path.abspath("..")
else:
    ROOT_DIR = os.getcwd()

if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

DATA_PATH = os.path.join(ROOT_DIR, "data", "train_test_split", "X_train.npy")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs", "generated_midis")
os.makedirs(OUTPUT_DIR, exist_ok=True)


from src.generation.midi_export import save_piano_roll_as_midi

print("ROOT_DIR:", ROOT_DIR)
print("DATA_PATH:", DATA_PATH)
print("DATA_PATH exists:", os.path.exists(DATA_PATH))


X_train = np.load(DATA_PATH)
print("X_train shape:", X_train.shape) 

def estimate_markov_model(X):
    """
    Estimate a first-order per-pitch Markov model from piano-roll windows.

    X shape: (N, T, P)
      N = number of windows
      T = time steps
      P = pitches (88)
    """
    N, T, P = X.shape

  
    init_prob = X[:, 0, :].mean(axis=0).astype(np.float32)  

   
    prev_states = X[:, :-1, :].reshape(-1, P)
    curr_states = X[:, 1:, :].reshape(-1, P)

   
    transition_prob = np.zeros((P, 2), dtype=np.float32)

    for pitch in range(P):
        for prev_state in [0, 1]:
            mask = prev_states[:, pitch] == prev_state
            count = np.sum(mask)

            if count == 0:
                transition_prob[pitch, prev_state] = 0.0
            else:
                transition_prob[pitch, prev_state] = curr_states[mask, pitch].mean()

    return init_prob, transition_prob

def generate_markov_sequence(init_prob, transition_prob, time_steps=128):
    """
    Generate a binary piano-roll sequence of shape (time_steps, 88)
    using the learned Markov model.
    """
    n_pitches = init_prob.shape[0]
    seq = np.zeros((time_steps, n_pitches), dtype=np.uint8)


    seq[0] = (np.random.rand(n_pitches) < init_prob).astype(np.uint8)


    for t in range(1, time_steps):
        prev_step = seq[t - 1]
        for pitch in range(n_pitches):
            p_on = transition_prob[pitch, prev_step[pitch]]
            seq[t, pitch] = 1 if np.random.rand() < p_on else 0

    return seq


init_prob, transition_prob = estimate_markov_model(X_train)
print("init_prob shape:", init_prob.shape)
print("transition_prob shape:", transition_prob.shape)


NUM_SAMPLES = 5

for i in range(NUM_SAMPLES):
    sample_seq = generate_markov_sequence(init_prob, transition_prob, time_steps=128)
    output_path = os.path.join(OUTPUT_DIR, f"markov_baseline_{i+1}.mid")
    save_piano_roll_as_midi(sample_seq, output_path, tempo=120)
    print("Saved:", output_path)

print("\nDone. Markov baseline MIDI files generated successfully.")

ROOT_DIR: a:\music-generation-unsupervised
DATA_PATH: a:\music-generation-unsupervised\data\train_test_split\X_train.npy
DATA_PATH exists: True
X_train shape: (35412, 128, 88)
init_prob shape: (88,)
transition_prob shape: (88, 2)
Saved: a:\music-generation-unsupervised\outputs\generated_midis\markov_baseline_1.mid
Saved: a:\music-generation-unsupervised\outputs\generated_midis\markov_baseline_2.mid
Saved: a:\music-generation-unsupervised\outputs\generated_midis\markov_baseline_3.mid
Saved: a:\music-generation-unsupervised\outputs\generated_midis\markov_baseline_4.mid
Saved: a:\music-generation-unsupervised\outputs\generated_midis\markov_baseline_5.mid

Done. Markov baseline MIDI files generated successfully.
